## 📊 Introduction

In today’s data-driven world, organizations rely heavily on data to make informed business decisions. However, raw data is often unstructured, inconsistent, and not directly usable for analysis. This project focuses on building an end-to-end data pipeline using Databricks and Apache Spark to transform raw e-commerce data into meaningful business insights.

The pipeline follows the Medallion Architecture, consisting of three layers: Bronze (raw data ingestion), Silver (data cleaning and transformation), and Gold (business-level insights). At each stage, the data is progressively refined to improve its quality and usability.

Using PySpark and SQL, the project performs key operations such as data cleaning, feature engineering, and aggregation to generate insights like top-selling products, revenue by city, and monthly sales trends.

This project demonstrates practical applications of data engineering and data analysis concepts, showcasing how large-scale data can be processed efficiently and converted into actionable insights for business decision-making.


In [0]:
df = spark.read.table("default.sample_superstore")

df.show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|           City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|       Claire Gute|   Consumer|United States|      Henderson|      Kentucky|      42420|  South|FUR-BO-10001798|   

In [0]:
df = spark.read.table("default.sample_superstore")

# Clean column names
df = df.toDF(*[col.lower().replace(" ", "_") for col in df.columns])

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("bronze_ecommerce")

# **1. Bronze Layer - Raw Data**

In [0]:
spark.read.table("bronze_ecommerce").show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|     customer_name|    segment|      country|           city|         state|postal_code| region|     product_id|       category|sub-category|        product_name|   sales|quantity|discount|  profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|       Claire Gute|   Consumer|United States|      Henderson|      Kentucky|      42420|  South|FUR-BO-10001798|   

# **2.Silver Layer - Cleaned Data**

**Step 1: Load Bronze**

In [0]:
df = spark.read.table("bronze_ecommerce")

**STEP 2: Drop null values**

In [0]:
df_clean = df.dropna()

**STEP 3: Convert date columns**

In [0]:
from pyspark.sql.functions import to_date

df_clean = df_clean.withColumn("order_date", to_date("order_date")) \
                   .withColumn("ship_date", to_date("ship_date"))

**STEP 4: Create new column (feature engineering)**

In [0]:
from pyspark.sql.functions import col

df_clean = df_clean.withColumn("total_sales", col("sales") * col("quantity"))

**STEP 5: Add delivery time**

In [0]:
from pyspark.sql.functions import datediff

df_clean = df_clean.withColumn("delivery_days", datediff("ship_date", "order_date"))

**STEP 6: Save as Silver table**

In [0]:
df_clean.write.format("delta").mode("overwrite").saveAsTable("silver_ecommerce")

**STEP 7: Verify**

In [0]:
spark.read.table("silver_ecommerce").show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+------------------+-------------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|     customer_name|    segment|      country|           city|         state|postal_code| region|     product_id|       category|sub-category|        product_name|   sales|quantity|discount|  profit|       total_sales|delivery_days|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+------------------+-------------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|       Claire Gu

# **3.Gold Layer (Business Insights)**

**STEP 1: Load Silver data**

In [0]:
df = spark.read.table("silver_ecommerce")

**STEP 2: Top Selling Products**

In [0]:
top_products = df.groupBy("product_name") \
                 .sum("quantity") \
                 .orderBy("sum(quantity)", ascending=False)

top_products.show()

+--------------------+-------------+
|        product_name|sum(quantity)|
+--------------------+-------------+
|             Staples|          215|
|     Staple envelope|          170|
|   Easy-staple paper|          150|
|Staples in misc. ...|           86|
|KI Adjustable-Hei...|           74|
|Storex Dura Pro B...|           71|
|Avery Non-Stick B...|           71|
|GBC Premium Trans...|           67|
|Situations Contou...|           64|
|Staple-based wall...|           62|
|Eldon Wave Desk A...|           61|
|      Staple remover|           61|
|Chromcraft Round ...|           61|
|Global Wood Trimm...|           59|
|Wilson Jones Turn...|           59|
|Kingston Digital ...|           57|
|Fellowes Officewa...|           55|
|Global High-Back ...|           54|
|           Xerox 226|           53|
|SAFCO Arco Foldin...|           53|
+--------------------+-------------+
only showing top 20 rows


**STEP 3: Revenue by City**

In [0]:
revenue_by_city = df.groupBy("city") \
                    .sum("total_sales") \
                    .orderBy("sum(total_sales)", ascending=False)

revenue_by_city.show()

+-------------+------------------+
|         city|  sum(total_sales)|
+-------------+------------------+
|New York City| 1263477.611000001|
|  Los Angeles| 873209.3220000012|
|      Seattle| 597615.2260000001|
| Philadelphia| 567773.9109999998|
|San Francisco| 541425.7119999998|
|      Houston| 306053.3467999997|
|      Detroit|270415.30799999996|
|    San Diego| 260573.2110000001|
| Jacksonville|241771.37999999992|
|  Springfield|        234805.284|
|      Chicago|224054.56500000012|
|     Columbus|187410.70900000006|
|  San Antonio|140664.50999999998|
|      Jackson|132556.51599999997|
|     Columbia|        119010.244|
|    Arlington|118006.08399999999|
|       Newark|114870.23299999998|
|    Lafayette|         112252.48|
|   Burlington|110336.73400000001|
|     Lakewood|109005.52100000002|
+-------------+------------------+
only showing top 20 rows


**STEP 4: Revenue by Category**

In [0]:
revenue_by_category = df.groupBy("category") \
                        .sum("total_sales") \
                        .orderBy("sum(total_sales)", ascending=False)

revenue_by_category.show()

+---------------+------------------+
|       category|  sum(total_sales)|
+---------------+------------------+
|     Technology|4080261.5249999864|
|      Furniture|3859215.2288999907|
|Office Supplies| 3548585.318000002|
+---------------+------------------+



**STEP 5: Monthly Sales Trend**

In [0]:
from pyspark.sql.functions import month

monthly_sales = df.withColumn("month", month("order_date")) \
                  .groupBy("month") \
                  .sum("total_sales") \
                  .orderBy("month")

monthly_sales.show()

+-----+------------------+
|month|  sum(total_sales)|
+-----+------------------+
|    1| 505197.0024000002|
|    2| 264645.5524000001|
|    3| 980843.9724000006|
|    4| 627581.3683999996|
|    5| 797874.1184999995|
|    6| 720871.2404999988|
|    7| 702224.0310000004|
|    8| 878110.9762000003|
|    9|1515795.0461000002|
|   10| 997496.4370999996|
|   11|1749999.8322000024|
|   12|      1747422.4947|
+-----+------------------+



**STEP 6: Save Gold Table**

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("gold_ecommerce")

**STEP 7: Verify**

In [0]:
spark.read.table("gold_ecommerce").show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+------------------+-------------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|     customer_name|    segment|      country|           city|         state|postal_code| region|     product_id|       category|sub-category|        product_name|   sales|quantity|discount|  profit|       total_sales|delivery_days|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+------------------+-------------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|       Claire Gu